In [1]:
from Bio import GenBank
from Bio import SeqIO
import json

In [2]:
d = {
    "label": "Mycobacterium tuberculosis (Mtb)",
    "length": 4411532,
    "genes": {},
    "reference": {}
}

In [3]:
## parse the genbank file
def parse_location(loc):
    def clean(loc):
        loc = loc.strip(">")
        loc = loc.strip("<")
        return loc

    if "complement" in loc:
        strand = 2
        loc = loc.split("(")[1].split(")")[0]
        [start, end] = loc.split("..")
    else:
        strand = 1
        start, end = loc.split("..")[0], loc.split("..")[1]

    return int(clean(start)), int(clean(end)), strand

with open("../data/h37Rv.gb") as handle:
    for record in GenBank.parse(handle):
        print(dir(record))
        for feature in record.features:
            if feature.key == "gene":
                if "order" not in feature.location:
                    for q in feature.qualifiers:
                        if "/gene" in q.key:
                            n = q.value.strip('"')
                            start, end, strand = parse_location(feature.location)
                            gene_d = {
                                "start": start,
                                "end": end,
                                "strand": strand
                                }
                            d["genes"][n] = gene_d

['BASE_FEATURE_FORMAT', 'BASE_FORMAT', 'GB_BASE_INDENT', 'GB_FEATURE_INDENT', 'GB_FEATURE_INTERNAL_INDENT', 'GB_INTERNAL_INDENT', 'GB_LINE_LENGTH', 'GB_OTHER_INTERNAL_INDENT', 'GB_SEQUENCE_INDENT', 'INTERNAL_FEATURE_FORMAT', 'INTERNAL_FORMAT', 'OTHER_INTERNAL_FORMAT', 'SEQUENCE_FORMAT', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_accession_line', '_base_count_line', '_comment_line', '_contig_line', '_db_source_line', '_dblink_line', '_definition_line', '_features_line', '_keywords_line', '_locus_line', '_nid_line', '_organism_line', '_origin_line', '_pid_line', '_project_line', '_segment_line', '_sequence_line', '_source_line', '_version_line', '_wgs_line', '_wgs_scafld_line', 'accession',

In [ ]:
## parse the fasta
with open("../data/H37Rv.fasta") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        d["reference"]["label"] = "H37Rv"
        d["reference"]["accession"] = "AL123456.3"
        d["reference"]["sequence"] = str(record.seq)

## NOPE
# with open("../data/amr.fasta") as handle:
#     for record in SeqIO.parse(handle, "fasta"):
#         d["reference"]["label"] = str(record.id)
#         d["reference"]["accession"] = str(record.id)
#         d["reference"]["sequence"] = str(record.seq)

In [5]:
with open("../rampart/protocols/md2026/genome.json", "w") as f:
    print(json.dumps(d, indent=4), file=f)